# Act 1 — Why cloud at all?

You want to run software for other people to use. You have two options. Buy servers, rack them in a building you cool and secure, and hire people to look after them — or rent that capacity by the hour from a company that already does all of that. Cloud is the second option, treated as a default.

Everything that follows comes from that one choice — what you give up, what you get, and how the trade plays out at scale.

## What cloud actually means

Cloud computing is the **on-demand delivery of compute, storage, networking, and dozens of other services over the internet**, paid for by what you actually consume. The provider owns the hardware, the buildings, the cooling, the security, the network. You request resources through an API; they appear in seconds; you pay only while they exist.

That single shift — from **owning** capacity to **renting** it — changes almost every architectural decision downstream.

## Traditional IT vs. Cloud

| | Traditional IT | Cloud |
|---|---|---|
| **Hardware** | Buy and own | Rent on-demand |
| **Capacity** | Over-provision for peak | Scale up/down in minutes |
| **Speed** | Weeks to provision | Seconds to provision |
| **Cost model** | Large upfront CapEx | Pay-as-you-go OpEx |
| **Maintenance** | Your team manages racks, power, cooling | Provider handles undifferentiated heavy lifting |
| **Global reach** | Slow, expensive | Deploy worldwide in minutes |

**Old pain points cloud removes:** paying for idle capacity, multi-week procurement, guessing capacity 18 months out, and running a physical data center.

## Three Service Models

Which layer you consume determines how much of the operational stack you still own.

| Model | Provider manages | You manage | GCP examples |
|---|---|---|---|
| **IaaS** — Infrastructure as a Service | Virtualization, networking, hardware | OS, runtime, app, data | Compute Engine, Persistent Disk, VPC |
| **PaaS** — Platform as a Service | + OS, + runtime | App and data only | Cloud Run, Cloud SQL, App Engine |
| **SaaS** — Software as a Service | The entire stack | Just your data / use the app | Google Workspace, Looker, Apigee |

Each step up the stack trades **flexibility** for **less operational burden**. GCP's centre of gravity sits firmly in PaaS — Cloud Run, BigQuery, Pub/Sub, Cloud SQL, GKE Autopilot — which is part of why teams that lean on managed services tend to like GCP.

## Deployment Models

| Model | What it is | When you see it |
|---|---|---|
| **Public** | Provider-owned, multi-tenant | Default for everything in this course |
| **Private** | Single-organization infrastructure | Banks, government, regulated workloads |
| **Hybrid** | Public + private connected over a dedicated link | Most large enterprises |
| **Multi-cloud** | More than one public provider | Vendor lock-in avoidance, niche service usage |

GCP's bet on hybrid is **Anthos** and **Google Distributed Cloud (GDC)** — mentioned later in this notebook and covered properly in the chapters where they earn their keep.

# Act 2 — Where does GCP run your code?

Look at a GCP map and you see regions, zones, multi-region locations, edge POPs, sovereign cloud variants. That is a lot of words for "where your code runs," and on a first read it looks like Google is collecting jargon.

It isn't. All of it falls out of three pressures the cloud has to handle at once:

- **Latency** — your users are spread around the world and the speed of light is fixed.
- **Sovereignty** — data has legal homes. EU customer data may need to stay in the EU; some workloads must stay inside one country entirely.
- **Failure isolation** — your service cannot go down when one building loses power or one network spine has a bad day.

Read this act as three pressures and their specialised answers, not a feature catalogue.

## GCP Global Infrastructure

GCP infrastructure is a set of nested layers. Each layer trades **breadth** for **proximity to the user**.

| Layer | What it is | Count (~2026) | Runs |
|---|---|---|---|
| **Region** | Independent geographic area with multiple data centers | 40+ | All regional GCP services |
| **Zone** | One or more discrete data centers within a region | 3+ per region | Zonal resources (VM instances, zonal disks) |
| **Multi-region location** | A logical wrapper spanning two or more regions | `us`, `eu`, `asia`, plus dual-regions | Cloud Storage buckets, BigQuery datasets, Spanner configs |
| **Edge POP / GFE** | Google Front End locations | 200+ globally | Cloud Load Balancing termination, Cloud CDN, Cloud DNS |
| **Sovereign Cloud variants** | Operated by/with a local partner under local sovereignty rules | Select countries | Subset of services, varies by partner |
| **Google Distributed Cloud (GDC)** | GCP hardware on your premises, air-gapped or connected | Anywhere you ship it | GKE, GCE, select services |

## Regions

A **Region** is a physical geographic area containing a cluster of Google data centers — Iowa, Belgium, Frankfurt, Mumbai, Tokyo. Each has a code: `us-central1`, `europe-west1`, `asia-south1`. The number is just an identifier; it doesn't count anything.

**Key property: independence.** Regions don't share power, networking, or control planes. A failure in one has no causal connection to another. Most GCP resources live *in* a region — a Cloud SQL instance, a regional Cloud Storage bucket, a subnet. To have a presence in another region you deploy there explicitly.

**Global services** are the exception, and the list is genuinely longer than AWS's: **IAM**, **Cloud DNS**, **Cloud Load Balancing** (the global external Application LB), **Cloud Armor**, and — the GCP-unique one — **VPC**. A single VPC spans every region; subnets inside it are regional. Coming from AWS, where a VPC is regional, this is the first conceptual surprise. We come back to it in notebook 06.

## Zones

A **zone** is one or more discrete data centers, each with its own power, cooling, and physical security.

- **At least three zones per region**, named by suffix: `us-central1-a`, `us-central1-b`, `us-central1-c`
- **Tens of miles apart** — far enough that a single fire/flood cannot take more than one
- **Connected by high-bandwidth, low-latency fiber** — network between them is fast and effectively local

**Why they exist:** running across two or three zones inside a region is the foundation of high availability on GCP. Regional Managed Instance Groups (MIGs), regional Cloud SQL HA, regional Persistent Disks — every "regional" resource is really "multi-zone within a region."

**The subtlety vs AWS:** GCP **subnets are regional**, not zonal. One subnet's IP range spans every zone in the region. Instances are zonal; the subnet they live in is not. This makes multi-zone deployment substantially simpler — you don't manage a subnet-per-AZ as you would on AWS.

## Multi-region locations

A **multi-region location** is a logical wrapper that spans two or more regions for storage redundancy. GCP exposes three big ones — `us`, `eu`, `asia` — plus a growing list of **dual-region** locations (e.g. `nam4` = Iowa + South Carolina) that give you the same picking-two-regions-yourself behaviour without the picking.

This is what you select when you create:

- A **multi-region Cloud Storage bucket** — data is replicated to multiple regions inside the location automatically.
- A **BigQuery dataset** — data lives in either a region (`us-central1`) or a multi-region (`us`, `eu`).
- A **multi-region Spanner config** — synchronous replication across at least three regions, with optional read-only replicas.

AWS and Azure don't have a direct equivalent. The closest is S3 Multi-Region Access Points or Cross-Region Replication, which are *configurations* on top of regional resources — multi-region in GCP is a *first-class location type*, picked at create time.

## Edge POPs and the Google Front End

Edge POPs are **not regions** — they don't run Compute Engine or BigQuery. They are smaller facilities deployed in many more cities than regions, whose only job is to terminate user connections close to the user and forward into the Google backbone.

The machine doing that termination is the **Google Front End (GFE)** — a globally distributed reverse proxy fleet. When you create a Global External Application Load Balancer, the public IP you get is **anycast**: every GFE on the planet advertises it. A user's TCP and TLS handshake terminate at the closest GFE; the request then flows over Google's own private network — not the public internet — to your backends.

This matters: your backends can be in a single region, and a user in São Paulo still gets a low-latency TLS handshake against the local GFE. It's also why Cloud Armor (the WAF) is applied *at the GFE*, not at the regional backend.

| Service | What it does at the edge |
|---|---|
| **Global External Application LB** | Anycast IP, TLS termination at GFE |
| **Cloud CDN** | Caches content at the GFE |
| **Cloud Armor** | WAF + DDoS filtering at the GFE |
| **Cloud DNS** | Anycast resolution from edge POPs |

## Sovereign Cloud and Distributed Cloud

Three options for when even an ordinary region is too far away — or when regulation forbids ordinary regions altogether.

| Option | Direction | Use case |
|---|---|---|
| **Sovereign Cloud** | Operated by/with a local partner under local sovereignty rules | EU sovereign workloads, regulated industries |
| **GDC Hosted** | Air-gapped GCP rack hardware in your facility | Defence, intelligence, top-secret workloads |
| **GDC Connected** | GCP rack hardware on your premises, connected back to a region | Strict data residency, ultra-low latency to local industrial systems |

These are not exam-blueprint headliners, but it's worth knowing they exist so you don't confuse them with Local Zones or Outposts on AWS — they are GCP's parallel answers to the same pressures.

# Act 3 — The resource hierarchy

Everything in GCP lives inside a tree. The tree is short, only four levels, but the tree shape is the single most important fact about GCP — every IAM binding, every Organization Policy constraint, every billing aggregation hangs off it.

If you skip this section, every later notebook will feel arbitrary. If you internalise it now, the rest of the course gets noticeably easier.

## Organization → Folder → Project → Resource

Four layers. The top three are containers; the bottom is the actual stuff.

- **Organization** — the root. One per company, tied to a Google Workspace / Cloud Identity domain. The Organization is where company-wide IAM and Org Policy live.
- **Folders** — optional intermediate containers. Use them to mirror business units, environments (`prod`, `dev`), or shared platforms. A folder can contain folders or projects.
- **Projects** — the unit of isolation for resources, IAM, billing, quotas, and APIs. Every resource lives in exactly one project. A project has a globally unique **ID** (chosen by you, e.g. `acme-payments-prod`), a numeric **number** (assigned by GCP), and a human-readable **name**.
- **Resources** — Compute Engine VMs, GCS buckets, BigQuery datasets, Cloud Run services. They sit *inside* a project; they cannot exist outside one.

**Compare:**

- **AWS Organizations / OUs / Accounts / Resources** — GCP **Folders ≈ AWS OUs**, GCP **Projects ≈ AWS Accounts**. The shapes line up; the units differ.
- **Azure Management Groups / Subscriptions / Resource Groups / Resources** — GCP **Folders ≈ Azure MGs**, GCP **Projects ≈ Azure Subscriptions**. There is no GCP equivalent of a Resource Group as a separate layer — projects do double duty as the IAM and resource boundary.

## The hierarchy at a glance

```
Organization: acme.com
├── Folder: platform
│   ├── Project: acme-shared-vpc
│   ├── Project: acme-logging
│   └── Project: acme-billing-export
├── Folder: workloads
│   ├── Folder: dev
│   │   ├── Project: acme-app-dev
│   │   └── Project: acme-data-dev
│   └── Folder: prod
│       ├── Project: acme-app-prod
│       └── Project: acme-data-prod
└── Folder: sandbox
    └── Project: alice-experiments
```

Real-world hierarchies use folders to model environments and business units; projects below them isolate IAM, billing, and quotas at a workload level.

## The IAM cascade — the single most important rule

An IAM binding at any node in the tree applies to **that node and everything beneath it**. Bindings are additive going down: a binding at the Organization grants access in every project; a binding at a folder grants access in every project under that folder; a binding at a project grants access in that project only.

The practical pattern:

- Grant **broad, infrequent** roles high in the tree — e.g. `roles/billing.viewer` to a finance group at the Organization.
- Grant **narrow, frequent** roles at the project — e.g. `roles/run.developer` to an app team at `acme-app-prod`.
- Never grant `roles/owner` at the Organization. Almost never grant it at a folder. Project owner is sometimes fine.

This cascade is what makes Folders worth the bother of setting up — they let you grant once, high, instead of N times across N projects.

**The contrast vs AWS:** AWS IAM does not cascade like this. SCPs apply across an OU, but identity policies are per-account. GCP's downward inheritance is the central design decision behind the hierarchy — and the central thing to get right in real environments. Notebook 02 is entirely about its mechanics.

# Act 4 — Owning what you own, and choosing a region

You have seen what cloud trades, where GCP runs your code, and the tree everything hangs from. Three small but load-bearing topics close out this notebook.

First — how do you actually talk to GCP? The console, the `gcloud` CLI, and the client libraries are three different doors into the same control plane.

Second — when something goes wrong, whose problem is it? Google owns part of the stack, you own the rest, and the line between them moves depending on the service.

Third — given a planet's worth of regions, which one do you actually pick for a new workload?

## How You Talk to GCP

Every interaction goes through the same control plane — three doors into it.

| Door | What it is | When to use |
|---|---|---|
| **Google Cloud Console** | Web UI in a browser | Learning a new service, one-off exploration |
| **`gcloud` CLI** | Commands from your terminal | Scripting, automation glue, ad-hoc ops |
| **Client libraries** (e.g. `google-cloud-*`) | Programmatic access from code | Application code, IaC, repeatable workflows |

Underneath all three: signed HTTPS requests to a regional or global API endpoint. The console is a web app calling those endpoints; the CLI is a thin wrapper; the SDK is a typed wrapper.

A fourth door is worth a mention: **Cloud Shell**, a free Linux VM in a browser with `gcloud` pre-installed and your credentials already loaded. The fastest way to try a command without configuring anything locally.

In [ ]:
# The google-cloud-compute SDK can list regions for a project once
# credentials are configured. Set them up with:
#   gcloud auth application-default login
from google.cloud import compute_v1

PROJECT_ID = "your-project-id"

client = compute_v1.RegionsClient()
for r in client.list(project=PROJECT_ID):
    print(f"{r.name:24s}  status={r.status}")

In [ ]:
# Zones in a single region. Useful to see the zone suffixes (a, b, c, ...)
# you'll later use in VM placement and regional MIG configuration.
#
# from google.cloud import compute_v1
# client = compute_v1.ZonesClient()
# for z in client.list(project=PROJECT_ID):
#     if z.region.endswith("/us-central1"):
#         print(z.name, "-", z.status)

## Shared Responsibility — Primer

| | Google is responsible for | You are responsible for |
|---|---|---|
| **Slogan** | Security **of** the cloud | Security **in** the cloud |
| **Always** | Physical buildings, hardware, hypervisors, networking, the services they operate | Your data, your IAM bindings, your firewall rules, your encryption key choices |
| **Compute Engine** | Hardware, hypervisor | Guest OS patching, application, firewall rules |
| **Cloud SQL** | Hardware, OS, database engine patching | Users, queries, backups configuration |
| **Cloud Storage** | Hardware, OS, storage durability | Data, bucket policies, access controls |
| **Cloud Run** | Hardware, OS, runtime, scaling | Container image, env config, IAM |

**The line moves up the stack as the service gets more managed — but responsibility for *what you put in* and *who you let see it* never goes away.** Cloud Run and BigQuery push that line further up than Compute Engine does; that's the GCP centre-of-gravity I mentioned earlier showing up in practice. Notebook 12 covers observability and governance in depth.

## Architecture Framework — Six Pillars

The lens Google publishes for reviewing any workload. Not a checklist — a vocabulary for trade-offs.

| Pillar | Goal | Concrete techniques |
|---|---|---|
| **Operational Excellence** | Run and improve systems to deliver value | IaC, observability, runbooks, post-mortems |
| **Security, Privacy & Compliance** | Protect data, systems, assets; meet regulatory obligations | IAM, CMEK, VPC-SC, audit logs, Security Command Center |
| **Reliability** | Perform correctly; recover from failure | Multi-zone, regional MIGs, regional databases, DR planning |
| **Cost Optimization** | Deliver value at the lowest price point | Right-sizing, CUDs, Spot, kill idle resources, billing exports |
| **Performance Optimization** | Right resources for the job, kept right as demand shifts | Right-sizing, the right database for the workload, GFE-fronted LB |
| **Sustainability** | Minimize environmental impact | Carbon-aware region choice, right-sizing, managed services |

Notebook 14 returns to this with concrete service mappings. The pillar names line up nearly one-for-one with AWS's Well-Architected Framework — only the merger of "Security" with "Privacy & Compliance" is GCP-specific.

## Choosing a Region

Four factors, in roughly this order of weight:

1. **Compliance & data residency** — some regulations require data to stay in a country or economic zone. Often non-negotiable.
2. **Latency** — closest region to your users wins. If users are spread across continents, design multi-region with a Global External Application LB in front (the GFE will terminate close to the user regardless of where your backends sit).
3. **Service availability** — newer services don't ship in every region simultaneously. Check the GCP locations page for every service you need before committing.
4. **Pricing & carbon footprint** — varies modestly between regions. GCP publishes a per-region **carbon-free energy percentage**; for sustainability-sensitive workloads, regions like `europe-north1` (Finland) and `us-central1` (Iowa) rank high. Pricing is the tiebreaker, not the driver.